# EDF Load via MountedEdfSourceAdapter

Load EDF recordings from a mounted folder or repo-local symlink using `MountedEdfSourceAdapter`, then inspect one selected recording with MNE and notebook-friendly tables.


## 1. Imports

Install missing packages in your environment if needed:

```bash
pip install mne pandas numpy matplotlib xarray ipywidgets mne-qt-browser PyQt6
```


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists() and (REPO_ROOT.parent / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.io.sources.mounted_edf_source_adapter import MountedEdfSourceAdapter

try:
    import mne
except ImportError as exc:
    raise ImportError("Install MNE first: pip install mne") from exc

try:
    import xarray as xr
except ImportError:
    xr = None

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 20)

mne.set_log_level("INFO")


## 2. Choose EDF Source

Point `SOURCE_DIR` at a mounted folder or repo-local symlink containing EDF files. The adapter emits complete recordings for ingestion, and this notebook also selects one EDF file from that source for direct MNE exploration.


In [ ]:
SOURCE_DIR = REPO_ROOT / "data" / "rawdata"/"sub-041_id-236"
EDF_PATTERN = "**/*.edf"
SOURCE_ID = "hypnose-eeg-raw-edf"
source_adapter = MountedEdfSourceAdapter(
    source_id=SOURCE_ID,
    source_dir=str(SOURCE_DIR),
    pattern=EDF_PATTERN,
)
source_adapter.open()

edf_files = sorted(SOURCE_DIR.glob(EDF_PATTERN))
if not edf_files:
    raise FileNotFoundError(f"No EDF files found under {SOURCE_DIR.resolve()} with pattern {EDF_PATTERN!r}")

EDF_PATH = edf_files[0]
EDF_PATH.resolve()


## 3. Preview Adapter Recording

The adapter emits complete `EEGRecording` objects plus a `SourceCursor`. Save the cursor after downstream processing when building resumable ingestion.


In [ ]:
recording_iter = source_adapter.recordings(start_cursor=None)
first_recording, first_cursor = next(recording_iter)

adapter_preview = {
    "source_id": first_recording.source_id,
    "session_id": first_recording.session_id,
    "timestamp_utc": first_recording.timestamp_utc,
    "sample_rate_hz": first_recording.sample_rate_hz,
    "channels": first_recording.channels,
    "n_channels": len(first_recording.samples),
    "n_samples_per_channel": len(first_recording.samples[0]) if first_recording.samples else 0,
    "source_file": first_recording.metadata["source_file"],
    "source_path": first_recording.metadata["source_path"],
    "duration_seconds": first_recording.metadata["duration_seconds"],
    "cursor_file": first_cursor.last_file,
}

pd.Series(adapter_preview)


## 4. Load EDF as MNE Raw

`raw` is the best primary format for EEG exploration in Python because it preserves channel metadata, sampling rate, annotations, and plotting utilities.

Use `preload=False` first for large EDF files. You can crop and load smaller sections later.


In [ ]:
raw = mne.io.read_raw_edf(
    EDF_PATH,
    preload=False,
    infer_types=True,
    verbose=True,
)

raw


## 5. Recording Summary


In [ ]:
summary = {
    "file": str(EDF_PATH),
    "n_channels": len(raw.ch_names),
    "sampling_rate_hz": raw.info["sfreq"],
    "n_samples": raw.n_times,
    "duration_seconds": raw.n_times / raw.info["sfreq"],
    "duration_hours": raw.n_times / raw.info["sfreq"] / 3600,
    "meas_date": raw.info.get("meas_date"),
}

pd.Series(summary)


In [ ]:
channel_table = pd.DataFrame({
    "channel": raw.ch_names,
    "type": raw.get_channel_types(),
})

channel_table


## 6. Annotations and Events

EDF files may include sleep stages, markers, or other annotations. This cell converts annotations to a table when present.


In [ ]:
annotations_df = pd.DataFrame({
    "onset_seconds": raw.annotations.onset,
    "duration_seconds": raw.annotations.duration,
    "description": raw.annotations.description,
})

annotations_df


## 7. Pick Channels and Time Window

Create a smaller loaded view for notebook exploration. Adjust `START_SECONDS`, `DURATION_SECONDS`, and `CHANNELS`.


In [ ]:
START_SECONDS = 0
DURATION_SECONDS = 60

# Use None for all channels, or provide names such as ["C3", "C4", "O1", "O2"].
CHANNELS = None

tmin = START_SECONDS
tmax = START_SECONDS + DURATION_SECONDS

raw_window = raw.copy().crop(tmin=tmin, tmax=tmax, include_tmax=False)

if CHANNELS is not None:
    raw_window.pick(CHANNELS)

raw_window.load_data()
raw_window


## 8. Convert Window to Pandas

Wide format is convenient for plotting and quick inspection: one row per time point, one column per channel.


In [ ]:
data, times = raw_window.get_data(return_times=True)

edf_wide_df = pd.DataFrame(data.T, columns=raw_window.ch_names)
edf_wide_df.insert(0, "time_seconds", times + START_SECONDS)

edf_wide_df.head()


In [ ]:
edf_long_df = edf_wide_df.melt(
    id_vars="time_seconds",
    var_name="channel",
    value_name="value",
)

edf_long_df.head()


## 9. Convert Window to Xarray

`xarray` is useful for labeled multidimensional data. The output has dimensions `channel` and `time_seconds`.


In [ ]:
if xr is None:
    print("xarray is not installed. Install with: pip install xarray")
else:
    edf_xr = xr.DataArray(
        data,
        dims=("channel", "time_seconds"),
        coords={
            "channel": raw_window.ch_names,
            "time_seconds": times + START_SECONDS,
        },
        attrs={
            "source_file": str(EDF_PATH),
            "sampling_rate_hz": raw_window.info["sfreq"],
            "units": "MNE default SI units, usually volts for EEG",
        },
        name="edf_signal",
    )
    display(edf_xr)


## 10. Quick Plots


In [ ]:
plot_channels = raw_window.ch_names[: min(8, len(raw_window.ch_names))]

fig, ax = plt.subplots(figsize=(14, 6))
channel_spacing = 5.0
yticks = []

for channel_index, channel in enumerate(plot_channels):
    values = edf_wide_df[channel].to_numpy()
    scale = np.nanstd(values) or 1.0
    offset = channel_index * channel_spacing
    ax.plot(edf_wide_df["time_seconds"], values / scale + offset, label=channel)
    yticks.append(offset)

ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Scaled amplitude with channel offsets")
ax.set_yticks(yticks)
ax.set_yticklabels(plot_channels)
ax.set_title("EDF signal preview")
plt.show()


In [ ]:
# MNE's built-in browser is often the best interactive view.
# The Qt backend opens a separate interactive browser window from Jupyter.
# 
try:
    get_ipython().run_line_magic("gui", "qt")
except NameError:
    pass

try:
    mne.viz.set_browser_backend("qt")
except Exception as exc:
    raise RuntimeError("Install the interactive MNE browser with: pip install mne-qt-browser PyQt6") from exc

channel_types = raw_window.get_channel_types()
mne_scalings = {}

for channel_type in sorted(set(channel_types)):
    picks = [idx for idx, kind in enumerate(channel_types) if kind == channel_type]
    values = raw_window.get_data(picks=picks)
    scale = np.nanpercentile(np.abs(values), 95) * 2
    mne_scalings[channel_type] = max(scale, 1e-12)

raw_window.plot(
    duration=min(30, DURATION_SECONDS),
    n_channels=min(20, len(raw_window.ch_names)),
    scalings=mne_scalings,
    butterfly=False,
    remove_dc=True,
    block=False,
)


## 11. Optional: Downsample for Faster Exploration

Downsampling is useful when plotting long windows or exporting compact notebook-friendly tables.


In [ ]:
TARGET_SFREQ = 100

raw_window_resampled = raw_window.copy().resample(TARGET_SFREQ)
resampled_data, resampled_times = raw_window_resampled.get_data(return_times=True)

edf_resampled_df = pd.DataFrame(resampled_data.T, columns=raw_window_resampled.ch_names)
edf_resampled_df.insert(0, "time_seconds", resampled_times + START_SECONDS)

edf_resampled_df.head()


## 12. Optional: Save Notebook-Friendly Outputs

Use these only for small windows or downsampled data. Saving full overnight EDF recordings as CSV can be very large.


In [ ]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stem = EDF_PATH.stem

# CSV for quick inspection.
csv_path = OUTPUT_DIR / f"{stem}_window_{START_SECONDS}s_{DURATION_SECONDS}s_resampled.csv"
edf_resampled_df.to_csv(csv_path, index=False)

# FIF preserves MNE metadata better than CSV.
fif_path = OUTPUT_DIR / f"{stem}_window_{START_SECONDS}s_{DURATION_SECONDS}s_raw.fif"
raw_window.save(fif_path, overwrite=True)

csv_path, fif_path
